# Análisis de Datos: Experimento de Eratóstenes

**Fecha de medición:** 20 de marzo de 2026 (Equinoccio de Primavera)  
**Autores:** Jean Lucca Buritica Cuervo, Daniel Soto Villada, Camilo Higuita  
**Coordenadas:** $\varphi = 6.2666°\,\text{N}$, $\lambda = 75.5694°\,\text{W}$ (Medellín, Colombia)  
**Longitud del gnomon:** $(102.0 \pm 0.1)\,\text{cm}$  
**Error en longitud de sombra:** $\pm 0.1\,\text{cm}$

---

## 1. Fundamento Teórico

### 1.1 El método de Eratóstenes

Eratóstenes de Cirene (c. 276–195 a.C.) estimó la circunferencia de la Tierra comparando los
ángulos de los rayos solares en **Alejandría** y **Siena** (Asuán) al mediodía del solsticio de verano:

- **Siena**: el Sol iluminaba el fondo de los pozos → ángulo cenital $= 0°$.
- **Alejandría**: un gnomon vertical proyectaba una sombra → ángulo cenital $\alpha \approx 7.2°$.

Asumiendo (i) que los rayos del Sol son paralelos (la distancia Tierra–Sol es enorme comparada
con el tamaño de la Tierra) y (ii) que la Tierra es esférica, la diferencia angular entre dos
sitios es igual a la diferencia de latitud:

$$\frac{\Delta\theta}{360°} = \frac{d}{C}
\implies
C = \frac{360° \cdot d}{\Delta\theta}$$

donde $d$ es la distancia entre los dos sitios a lo largo del meridiano y $C$ la circunferencia
terrestre.

### 1.2 Ventaja del equinoccio

En el **equinoccio de primavera** (20 de marzo de 2026), la declinación solar es $\delta = 0°$:
el Sol se encuentra exactamente sobre el ecuador. En consecuencia, al **mediodía solar**:

$$\theta_z = \varphi \quad \text{(ángulo cenital = latitud del observador)}$$

El ecuador actúa como la "ciudad de Siena" (ángulo cenital $= 0°$), y Medellín
($\varphi \approx 6.27°\,\text{N}$) como el segundo punto de observación. No se necesita
coordinar mediciones simultáneas con otra ciudad.

### 1.3 Cálculo del ángulo cenital

Si $L$ es la longitud de la sombra del gnomon y $H$ su altura:

$$\boxed{\theta_z = \arctan\!\left(\frac{L}{H}\right)}$$

Al mediodía solar, $L = L_{\min}$ y por tanto $\theta_z = \varphi$ (en el equinoccio).

### 1.4 Radio y circunferencia terrestres

Para calcular $d$ de forma **independiente** de $R_\oplus$, se usa el valor estándar de la
longitud de un grado de arco meridional en la región ecuatorial derivado del elipsoide WGS-84:
$M_1 = 110.574\;\text{km/°}$.  Así:

$$d = \varphi_{\text{deg}} \times M_1$$

Y luego:

$$C = \frac{360° \cdot d}{\theta_z}
\qquad
R_\oplus = \frac{d}{\theta_z\;[\text{rad}]}$$

### 1.5 Propagación de errores

Para una función $f(x_1, x_2, \ldots)$ con variables independientes:

$$\sigma_f = \sqrt{\sum_i \left(\frac{\partial f}{\partial x_i}\right)^2 \sigma_{x_i}^2}$$

**Ángulo cenital** $\theta_z = \arctan(L/H)$:

$$\frac{\partial \theta_z}{\partial L} = \frac{H}{H^2+L^2},
\qquad
\frac{\partial \theta_z}{\partial H} = \frac{-L}{H^2+L^2}$$

$$\boxed{\sigma_{\theta_z} = \frac{\sqrt{H^2\,\sigma_L^2 + L^2\,\sigma_H^2}}{H^2+L^2}}$$

**Radio terrestre** $R_\oplus = d\,/\,\theta_{z,\text{rad}}$ (con $d$ conocida con precisión GPS):

$$\sigma_{R_\oplus} = \frac{d}{\theta_{z,\text{rad}}^2}\,\sigma_{\theta_z}
= R_\oplus\,\frac{\sigma_{\theta_z}}{\theta_{z,\text{rad}}}$$

**Circunferencia** $C = 2\pi R_\oplus$:

$$\sigma_C = 2\pi\,\sigma_{R_\oplus}$$

In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit

# Configuracion global de graficas
plt.rcParams.update({
    'figure.figsize': (11, 5),
    'font.size': 12,
    'axes.grid': True,
    'grid.alpha': 0.35,
    'lines.linewidth': 2,
    'errorbar.capsize': 4,
})
print("Librerias importadas correctamente.")

## 2. Carga y descripción de los datos

Se dispone de dos archivos registrados el 20 de marzo de 2026 en Medellín:

| Archivo | Columnas | Puntos |
|:---|:---|:---:|
| `Datos20260320.dat` | `HH:MM  sombra(cm)` | 4 |
| `../DatosLongitudSombra.dat` | `sombra(cm)  HH:MM:SS` | 7 |

El gnomon tiene longitud $(H = 102.0 \pm 0.1)\,\text{cm}$ y la incertidumbre declarada en la
longitud de la sombra es $\sigma_L = 0.1\,\text{cm}$.

In [ ]:
# ── Parametros del experimento ──────────────────────────────────────────────
H    = 102.0   # Longitud del gnomon [cm]
sH   = 0.1     # Error en la longitud del gnomon [cm]
sL   = 0.1     # Error en la longitud de la sombra [cm]

# Coordenadas geograficas (GPS)
phi_deg = 6.266626540758183    # Latitud [grados N]
lam_deg = -75.56943558809242   # Longitud [grados W]

# ── Funcion auxiliar: HH:MM[:SS] -> minutos desde medianoche ────────────────
def hms_to_min(s):
    parts = s.strip().split(':')
    h, m = int(parts[0]), int(parts[1])
    sec  = float(parts[2]) if len(parts) == 3 else 0.0
    return h * 60.0 + m + sec / 60.0

def min_to_hhmmss(m):
    total_s = int(round(m * 60))
    h  = total_s // 3600
    mn = (total_s % 3600) // 60
    s  = total_s % 60
    return f"{h:02d}:{mn:02d}:{s:02d}"

# ── Datos20260320.dat (formato: HH:MM  sombra_cm) ───────────────────────────
t1_list, L1_list = [], []
with open('Datos20260320.dat') as f:
    for line in f:
        cols = line.strip().split()
        if len(cols) >= 2:
            t1_list.append(hms_to_min(cols[0]))
            L1_list.append(float(cols[1]))

t1 = np.array(t1_list)
L1 = np.array(L1_list)
idx = np.argsort(t1);  t1, L1 = t1[idx], L1[idx]

# ── DatosLongitudSombra.dat (formato: sombra_cm  HH:MM:SS) ──────────────────
t2_list, L2_list = [], []
with open('../DatosLongitudSombra.dat') as f:
    for line in f:
        cols = line.strip().split()
        if len(cols) >= 2:
            t2_list.append(hms_to_min(cols[1]))
            L2_list.append(float(cols[0]))

t2 = np.array(t2_list)
L2 = np.array(L2_list)
idx2 = np.argsort(t2);  t2, L2 = t2[idx2], L2[idx2]

# ── Imprimir tablas ──────────────────────────────────────────────────────────
print("=" * 40)
print("  Datos20260320.dat")
print(f"  {'#':>3}  {'Hora':>10}  {'Sombra (cm)':>12}")
print("-" * 40)
for i, (t, L) in enumerate(zip(t1, L1), 1):
    print(f"  {i:>3}  {min_to_hhmmss(t):>10}  {L:>12.1f}")

print()
print("=" * 40)
print("  DatosLongitudSombra.dat")
print(f"  {'#':>3}  {'Hora':>10}  {'Sombra (cm)':>12}")
print("-" * 40)
for i, (t, L) in enumerate(zip(t2, L2), 1):
    print(f"  {i:>3}  {min_to_hhmmss(t):>10}  {L:>12.1f}")

## 3. Visualización de los datos

A continuación se grafica la longitud de la sombra en función del tiempo para ambos conjuntos.
El **mínimo de la curva** corresponde al mediodía solar, momento en que el Sol alcanza su máxima
elevación y la sombra del gnomon es más corta.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# ── Panel A ──────────────────────────────────────────────────────────────────
ax1.errorbar(t1, L1, yerr=sL, fmt='o', color='steelblue',
             ms=7, label='Datos A', zorder=3)
ax1.set_xlabel('Tiempo (min desde medianoche)')
ax1.set_ylabel('Longitud de la sombra (cm)')
ax1.set_title('Datos20260320.dat  (Conjunto A)')
ax1.legend()
for t, L in zip(t1, L1):
    h, m = int(t)//60, int(t)%60
    ax1.annotate(f'{h:02d}:{m:02d}', (t, L),
                 textcoords='offset points', xytext=(6, 4), fontsize=9)

# ── Panel B ──────────────────────────────────────────────────────────────────
ax2.errorbar(t2, L2, yerr=sL, fmt='s', color='tomato',
             ms=7, label='Datos B', zorder=3)
ax2.set_xlabel('Tiempo (min desde medianoche)')
ax2.set_ylabel('Longitud de la sombra (cm)')
ax2.set_title('DatosLongitudSombra.dat  (Conjunto B)')
ax2.legend()
for t, L in zip(t2, L2):
    h, m = int(t)//60, int(t)%60
    ax2.annotate(f'{h:02d}:{m:02d}', (t, L),
                 textcoords='offset points', xytext=(6, 4), fontsize=9)

plt.suptitle('Longitud de la sombra del gnomon vs. tiempo (20-mar-2026)',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 4. Determinación del mediodía solar mediante ajuste parabólico

Cerca del mediodía solar la sombra sigue una parábola:

$$L(t) = a\,(t - t_0)^2 + L_{\min}$$

donde $t_0$ es la hora del mediodía solar y $L_{\min}$ la longitud mínima de la sombra.
Se ajusta esta función con `scipy.optimize.curve_fit` (mínimos cuadrados no lineales con
ponderación por los errores de medición).

### Mediodía solar teórico

A partir de las coordenadas del sitio se puede predecir la hora aproximada del mediodía solar:

| Concepto | Valor |
|:---|:---|
| Longitud geográfica | $\lambda = -75.5694°\,\text{W}$ |
| Meridiano UTC−5 | $75°\,\text{W}$ |
| Corrección por longitud | $\Delta t_\lambda = (75.5694 - 75)/15 \times 60 \approx +2.3\,\text{min}$ |
| Ecuación del tiempo (20 mar) | $E_T \approx -7.5\,\text{min}$ (Sol retrasado respecto al tiempo medio) |
| **Mediodía solar aparente** | $12{:}00 + 2{:}17 - (-7{:}30) \approx \mathbf{12{:}09{:}47}$ hora local |

Este valor concuerda con el mínimo observado experimentalmente (~12:10–12:12).

In [ ]:
# ── Ajuste parabolico: Conjunto A ───────────────────────────────────────────
def parabola(t, a, t0, Lmin):
    return a * (t - t0)**2 + Lmin

# Valores iniciales para el solver
p0_A = [0.02, t1[np.argmin(L1)], np.min(L1)]

popt_A, pcov_A = curve_fit(
    parabola, t1, L1, p0=p0_A,
    sigma=np.full(len(L1), sL), absolute_sigma=True
)
a_A, t0_A, Lmin_A = popt_A
sLmin_A = np.sqrt(pcov_A[2, 2])   # sigma de Lmin del ajuste
st0_A   = np.sqrt(pcov_A[1, 1])   # sigma de t0

# Chi-cuadrado reducido
res_A   = L1 - parabola(t1, *popt_A)
chi2_A  = np.sum((res_A / sL)**2)
dof_A   = len(L1) - 3
chi2r_A = chi2_A / dof_A

# ── Grafica ──────────────────────────────────────────────────────────────────
t_fine = np.linspace(t1.min() - 3, t1.max() + 3, 500)

fig, (ax, ax_res) = plt.subplots(1, 2, figsize=(14, 5))

ax.errorbar(t1, L1, yerr=sL, fmt='o', color='steelblue', ms=7,
            label='Datos A', zorder=5)
ax.plot(t_fine, parabola(t_fine, *popt_A), '-', color='navy',
        label='Ajuste parabolico')
ax.axvline(t0_A, color='gray', ls='--', alpha=0.7,
           label=f'Mediodia solar: {min_to_hhmmss(t0_A)}')
ax.axhline(Lmin_A, color='green', ls=':', alpha=0.8,
           label=f'Lmin = {Lmin_A:.3f} +/- {sLmin_A:.3f} cm')
ax.set_xlabel('Tiempo (min desde medianoche)')
ax.set_ylabel('Longitud de la sombra (cm)')
ax.set_title('Ajuste parabolico — Conjunto A')
ax.legend(fontsize=10)

# Residuos
ax_res.axhline(0, color='k', lw=1)
ax_res.errorbar(t1, res_A, yerr=sL, fmt='o', color='steelblue', ms=7)
ax_res.set_xlabel('Tiempo (min desde medianoche)')
ax_res.set_ylabel('Residuo (cm)')
ax_res.set_title(f'Residuos — chi2_red = {chi2r_A:.2f}')

plt.tight_layout()
plt.show()

print(f"Resultados del ajuste parabolico (Conjunto A):")
print(f"  Mediodia solar : {min_to_hhmmss(t0_A)} +/- {st0_A*60:.1f} s")
print(f"  L_min          = ({Lmin_A:.4f} +/- {sLmin_A:.4f}) cm")
print(f"  a              = ({a_A:.5f} +/- {np.sqrt(pcov_A[0,0]):.5f}) cm/min^2")
print(f"  chi2_red       = {chi2r_A:.3f}  (gl = {dof_A})")

In [ ]:
# ── Ajuste parabolico: Conjunto B ───────────────────────────────────────────
# Intento 1: ajuste con todos los 7 puntos
p0_B_all = [0.05, t2[np.argmin(L2)], np.min(L2)]
popt_B_all, pcov_B_all = curve_fit(
    parabola, t2, L2, p0=p0_B_all,
    sigma=np.full(len(L2), sL), absolute_sigma=True
)
a_B_all = popt_B_all[0]

print("Ajuste con todos los puntos del Conjunto B:")
print(f"  a = {a_B_all:.5f} cm/min^2  -->  {'abre hacia arriba (valido)' if a_B_all > 0 else 'abre hacia ABAJO (invalido fisicamente)'}")
print()

# Intento 2: ajuste con los 3 puntos mas cercanos al minimo observado
# (ultima region descendente-mínimo-ascendente: 12:09:29, 12:12:12, 12:14:40)
mask_B = t2 >= 729.0   # puntos desde 12:09 en adelante
t2_sub = t2[mask_B]
L2_sub = L2[mask_B]

p0_B_sub = [0.05, t2_sub[np.argmin(L2_sub)], np.min(L2_sub)]
popt_B, pcov_B = curve_fit(
    parabola, t2_sub, L2_sub, p0=p0_B_sub,
    sigma=np.full(len(L2_sub), sL), absolute_sigma=True
)
a_B, t0_B, Lmin_B = popt_B
sLmin_B = np.sqrt(pcov_B[2, 2])
st0_B   = np.sqrt(pcov_B[1, 1])

res_B   = L2_sub - parabola(t2_sub, *popt_B)
chi2_B  = np.sum((res_B / sL)**2)
dof_B   = len(t2_sub) - 3
# dof_B = 0 con 3 puntos → ajuste exacto
chi2r_B = chi2_B / dof_B if dof_B > 0 else float('nan')

print("Ajuste con los 3 puntos cercanos al minimo (12:09–12:14):")
print(f"  a = {a_B:.5f} cm/min^2  -->  {'abre hacia arriba (valido)' if a_B > 0 else 'invalido'}")

# ── Grafica completa ─────────────────────────────────────────────────────────
t_fine2 = np.linspace(t2.min() - 2, t2.max() + 2, 500)
t_fine2_sub = np.linspace(t2_sub.min() - 2, t2_sub.max() + 2, 300)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Panel izquierdo: datos + ajuste 7 puntos (invalido) + ajuste 3 puntos (valido)
ax = axes[0]
ax.errorbar(t2, L2, yerr=sL, fmt='s', color='tomato', ms=7, zorder=5, label='Datos B (7 puntos)')
ax.errorbar(t2_sub, L2_sub, yerr=sL, fmt='D', color='purple', ms=9, zorder=6,
            label='Puntos cercanos al minimo (3 pts)')
ax.plot(t_fine2, parabola(t_fine2, *popt_B_all), '--', color='gray', alpha=0.7,
        label=f'Ajuste 7 pts  (a={a_B_all:.4f}, invalido)')
ax.plot(t_fine2_sub, parabola(t_fine2_sub, *popt_B), '-', color='darkred', lw=2,
        label=f'Ajuste 3 pts  (a={a_B:.4f})')
ax.axvline(t0_B, color='gray', ls='--', alpha=0.6,
           label=f'Mediodia solar: {min_to_hhmmss(t0_B)}')
ax.axhline(Lmin_B, color='green', ls=':', alpha=0.8,
           label=f'Lmin = {Lmin_B:.3f} +/- {sLmin_B:.3f} cm')
ax.set_xlabel('Tiempo (min desde medianoche)')
ax.set_ylabel('Longitud de la sombra (cm)')
ax.set_title('Ajuste parabolico — Conjunto B')
ax.legend(fontsize=8.5)

# Etiquetar puntos atipicos
for t, L in zip(t2, L2):
    h, m = int(t)//60, int(t)%60
    ax.annotate(f'{h:02d}:{m:02d}', (t, L),
                textcoords='offset points', xytext=(5, 4), fontsize=8)

# Panel derecho: residuos del ajuste de 3 puntos (siempre 0 para 3 pts)
ax2 = axes[1]
ax2.axhline(0, color='k', lw=1)
ax2.errorbar(t2_sub, res_B, yerr=sL, fmt='D', color='purple', ms=8)
ax2.set_xlabel('Tiempo (min desde medianoche)')
ax2.set_ylabel('Residuo (cm)')
title_chi = f'Residuos — ajuste 3 puntos (exacto, gl={dof_B})'
ax2.set_title(title_chi)

plt.tight_layout()
plt.show()

print()
print(f"Resultados del ajuste parabolico — Conjunto B (3 puntos cercanos al minimo):")
print(f"  Mediodia solar : {min_to_hhmmss(t0_B)} +/- {st0_B*60:.1f} s")
print(f"  L_min          = ({Lmin_B:.4f} +/- {sLmin_B:.4f}) cm")
print(f"  a              = ({a_B:.5f} +/- {np.sqrt(pcov_B[0,0]):.5f}) cm/min^2")
print()
print("NOTA: El ajuste con los 7 puntos produce a < 0 (parabola invertida),")
print("indicando que los puntos de las 11:58 y 12:07 son outliers.")
print("Se usan los 3 puntos 12:09–12:14 (region mas confiable) para la estimacion.")

## 5. Cálculo del ángulo cenital y propagación de errores

A partir de $L_{\min}$ obtenido del ajuste parabólico se calcula el ángulo cenital del Sol
al mediodía solar:

$$\theta_z = \arctan\!\left(\frac{L_{\min}}{H}\right)$$

con su incertidumbre completa (incluyendo el aporte de $\sigma_H$ y $\sigma_{L_{\min}}$):

$$\sigma_{\theta_z} = \frac{\sqrt{H^2\,\sigma_{L_{\min}}^2 + L_{\min}^2\,\sigma_H^2}}{H^2 + L_{\min}^2}$$

El valor esperado (equinoccio) es igual a la latitud medida por GPS: $\varphi = 6.2666°$.

In [ ]:
# ── Angulo cenital: Conjunto A ──────────────────────────────────────────────
L_A  = Lmin_A
sLA  = sLmin_A

theta_A_rad = np.arctan(L_A / H)
theta_A_deg = np.degrees(theta_A_rad)

# Derivadas parciales para propagacion de errores
dtheta_dL_A = H / (H**2 + L_A**2)
dtheta_dH_A = -L_A / (H**2 + L_A**2)

stheta_A_rad = np.sqrt((dtheta_dL_A * sLA)**2 + (dtheta_dH_A * sH)**2)
stheta_A_deg = np.degrees(stheta_A_rad)

# ── Angulo cenital: Conjunto B ──────────────────────────────────────────────
L_B  = Lmin_B
sLB  = sLmin_B

theta_B_rad = np.arctan(L_B / H)
theta_B_deg = np.degrees(theta_B_rad)

dtheta_dL_B = H / (H**2 + L_B**2)
dtheta_dH_B = -L_B / (H**2 + L_B**2)

stheta_B_rad = np.sqrt((dtheta_dL_B * sLB)**2 + (dtheta_dH_B * sH)**2)
stheta_B_deg = np.degrees(stheta_B_rad)

# ── Contribucion relativa de cada fuente de error (Conjunto A) ───────────────
contrib_L_A = (dtheta_dL_A * sLA)**2
contrib_H_A = (dtheta_dH_A * sH)**2
total_A     = contrib_L_A + contrib_H_A

print("=" * 60)
print("  Contribucion relativa de cada fuente de error (Conjunto A)")
print("=" * 60)
print(f"  sigma_L  →  {contrib_L_A/total_A*100:.1f}% de la varianza total")
print(f"  sigma_H  →  {contrib_H_A/total_A*100:.1f}% de la varianza total")
print()

# ── Resultados ────────────────────────────────────────────────────────────────
print("=" * 60)
print("  Angulo cenital al mediodia solar")
print("=" * 60)
print(f"  {'Parametro':<28} {'Conjunto A':>12}  {'Conjunto B':>12}")
print("-" * 60)
print(f"  {'L_min (cm)':<28} {L_A:>8.4f}      {L_B:>8.4f}")
print(f"  {'sigma_L_min (cm)':<28} {sLA:>8.4f}      {sLB:>8.4f}")
print(f"  {'theta_z (grados)':<28} {theta_A_deg:>8.4f}      {theta_B_deg:>8.4f}")
print(f"  {'sigma_theta_z (grados)':<28} {stheta_A_deg:>8.4f}      {stheta_B_deg:>8.4f}")
print(f"  {'theta_z (rad)':<28} {theta_A_rad:>8.6f}      {theta_B_rad:>8.6f}")
print(f"  {'sigma_theta_z (rad)':<28} {stheta_A_rad:>8.6f}      {stheta_B_rad:>8.6f}")
print(f"  {'Latitud GPS phi (grados)':<28} {phi_deg:>8.4f}      {phi_deg:>8.4f}")
print(f"  {'|theta_z - phi| (grados)':<28} {abs(theta_A_deg-phi_deg):>8.4f}      {abs(theta_B_deg-phi_deg):>8.4f}")
print(f"  {'Error relativo en theta_z':<28} {abs(theta_A_deg-phi_deg)/phi_deg*100:>7.2f}%     {abs(theta_B_deg-phi_deg)/phi_deg*100:>7.2f}%")

# ── Grafica comparativa ──────────────────────────────────────────────────────
labels = ['Conjunto A', 'Conjunto B', 'Referencia GPS']
valores = [theta_A_deg, theta_B_deg, phi_deg]
errores = [stheta_A_deg, stheta_B_deg, 0.0]
colores = ['steelblue', 'tomato', 'green']

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(labels, valores, yerr=errores, capsize=7,
              color=colores, alpha=0.72, edgecolor='k')
ax.set_ylabel('Angulo cenital al mediodia solar (grados)')
ax.set_title('Angulo cenital medido vs. latitud GPS (referencia)')
y_max = max(v + e for v, e in zip(valores, errores)) * 1.18
ax.set_ylim(0, y_max)
for bar, val, err in zip(bars, valores, errores):
    ax.text(bar.get_x() + bar.get_width()/2, val + err + 0.06,
            f'{val:.3f}°', ha='center', va='bottom', fontsize=11, fontweight='bold')
plt.tight_layout()
plt.show()

## 6. Cálculo del radio y circunferencia de la Tierra

### 6.1 Distancia meridional

La distancia a lo largo del meridiano desde el ecuador hasta la latitud $\varphi$ se calcula
de dos maneras para verificar la consistencia:

**Método 1 – Aproximación lineal:**
$$d_1 = \varphi_{\text{deg}} \times M_1 = 6.2666° \times 110.574\;\text{km/°}$$

**Método 2 – Integración del elipsoide WGS-84:**
$$d_2 = a(1-e^2) \int_0^{\varphi} \frac{d\phi}{\left(1 - e^2 \sin^2\phi\right)^{3/2}}$$

con $a = 6378.137\;\text{km}$ y $e^2 = 0.00669438$.

### 6.2 Radio y circunferencia

$$R_\oplus = \frac{d}{\theta_{z,\text{rad}}}
\qquad
\sigma_{R_\oplus} = R_\oplus\,\frac{\sigma_{\theta_z}}{\theta_{z,\text{rad}}}
\qquad
C = 2\pi R_\oplus
\qquad
\sigma_C = 2\pi\,\sigma_{R_\oplus}$$

In [ ]:
# ── Distancia meridional ─────────────────────────────────────────────────────
M1 = 110.574            # km por grado de latitud (ecuatorial, WGS-84)
d_lineal = phi_deg * M1

# Integracion numerica — elipsoide WGS-84
a_WGS = 6378.137        # semieje mayor [km]
e2    = 0.00669438      # excentricidad al cuadrado

phi_arr = np.linspace(0.0, np.radians(phi_deg), 50000)
integrando = a_WGS * (1.0 - e2) / (1.0 - e2 * np.sin(phi_arr)**2)**1.5
d_WGS84 = np.trapezoid(integrando, phi_arr)

print(f"Distancia meridional (M1 x phi)  : d = {d_lineal:.4f} km")
print(f"Distancia meridional (WGS-84)    : d = {d_WGS84:.4f}  km")
print(f"Diferencia                        : {abs(d_lineal-d_WGS84):.4f} km  "
      f"({abs(d_lineal-d_WGS84)/d_WGS84*100:.3f}%)")
print()

# Usar d_WGS84 (mas preciso)
d = d_WGS84

# ── Valores de referencia ─────────────────────────────────────────────────────
R_ref = 6371.0                  # radio medio de la Tierra [km]
C_ref = 2 * np.pi * R_ref

# ── Radio y circunferencia: Conjunto A ───────────────────────────────────────
R_A = d / theta_A_rad
C_A = 2 * np.pi * R_A
sR_A = R_A * (stheta_A_rad / theta_A_rad)
sC_A = 2 * np.pi * sR_A
err_rel_A = (R_A - R_ref) / R_ref * 100

# ── Radio y circunferencia: Conjunto B ───────────────────────────────────────
R_B = d / theta_B_rad
C_B = 2 * np.pi * R_B
sR_B = R_B * (stheta_B_rad / theta_B_rad)
sC_B = 2 * np.pi * sR_B
err_rel_B = (R_B - R_ref) / R_ref * 100

# ── Imprimir resultados ───────────────────────────────────────────────────────
print("=" * 68)
print("  RESULTADOS FINALES")
print("=" * 68)
print(f"  {'Cantidad':<32} {'Conj. A':>14}  {'Conj. B':>14}")
print("-" * 68)
print(f"  {'theta_z (grados)':<32} "
      f"{theta_A_deg:>7.4f}+/-{stheta_A_deg:.4f}  "
      f"{theta_B_deg:>7.4f}+/-{stheta_B_deg:.4f}")
print(f"  {'d meridional (km)':<32} {d:>14.3f}  {d:>14.3f}")
print(f"  {'R_tierra (km)':<32} "
      f"{R_A:>7.1f}+/-{sR_A:.1f}  "
      f"{R_B:>7.1f}+/-{sR_B:.1f}")
print(f"  {'C_tierra (km)':<32} "
      f"{C_A:>7.1f}+/-{sC_A:.1f}  "
      f"{C_B:>7.1f}+/-{sC_B:.1f}")
print(f"  {'R aceptado (km)':<32} {R_ref:>14.1f}  {R_ref:>14.1f}")
print(f"  {'Error relativo en R':<32} {err_rel_A:>12.2f}%  {err_rel_B:>12.2f}%")
print("=" * 68)

# ── Grafica comparativa ──────────────────────────────────────────────────────
labels_R = [f'Conj. A\n{R_A:.0f} km', f'Conj. B\n{R_B:.0f} km',
            f'Referencia\n{R_ref:.0f} km']
vals_R   = [R_A, R_B, R_ref]
errs_R   = [sR_A, sR_B, 0.0]
cols_R   = ['steelblue', 'tomato', 'green']

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(labels_R, vals_R, yerr=errs_R, capsize=7,
              color=cols_R, alpha=0.72, edgecolor='k')
ax.set_ylabel('Radio terrestre (km)')
ax.set_title('Radio de la Tierra medido vs. valor de referencia')
y_range = max(vals_R) - min(vals_R)
margin  = max(y_range * 0.5, 300)
ax.set_ylim(min(vals_R) - margin, max(vals_R) + margin * 1.5)
for bar, val, err in zip(bars, vals_R, errs_R):
    ax.text(bar.get_x() + bar.get_width()/2, val + err + 20,
            f'{val:.0f}', ha='center', va='bottom',
            fontsize=11, fontweight='bold')
plt.tight_layout()
plt.show()

## 7. Tabla resumen de resultados

In [ ]:
# Tabla resumen completa
print("=" * 74)
print(f"  {'TABLA RESUMEN DE RESULTADOS':^70}")
print("=" * 74)
header = f"  {'Parametro':<36} {'Conj. A':>16}  {'Conj. B':>16}"
print(header)
print("-" * 74)

rows = [
    ("L_min (cm)",
     f"{L_A:.3f} +/- {sLA:.3f}",
     f"{L_B:.3f} +/- {sLB:.3f}"),
    ("theta_z (grados)",
     f"{theta_A_deg:.4f} +/- {stheta_A_deg:.4f}",
     f"{theta_B_deg:.4f} +/- {stheta_B_deg:.4f}"),
    ("theta_z (rad)",
     f"{theta_A_rad:.5f} +/- {stheta_A_rad:.5f}",
     f"{theta_B_rad:.5f} +/- {stheta_B_rad:.5f}"),
    ("R_tierra (km)",
     f"{R_A:.1f} +/- {sR_A:.1f}",
     f"{R_B:.1f} +/- {sR_B:.1f}"),
    ("C_tierra (km)",
     f"{C_A:.1f} +/- {sC_A:.1f}",
     f"{C_B:.1f} +/- {sC_B:.1f}"),
    ("Error relativo en R",
     f"{err_rel_A:+.2f}%",
     f"{err_rel_B:+.2f}%"),
    ("chi2_red del ajuste",
     f"{chi2r_A:.3f}",
     "N/A (ajuste exacto, 3 pts)"),
]
for name, valA, valB in rows:
    print(f"  {name:<36} {valA:>16}  {valB:>16}")

print("-" * 74)
print(f"  {'Latitud GPS phi (grados)':<36} {phi_deg:>16.6f}")
print(f"  {'Radio medio aceptado (km)':<36} {R_ref:>16.1f}")
print(f"  {'Circunferencia aceptada (km)':<36} {C_ref:>16.1f}")
print(f"  {'Distancia meridional d (km, WGS-84)':<36} {d:>16.4f}")
print("=" * 74)

## 8. Interpretación de los resultados

### 8.1 Calidad del ajuste

**Conjunto A (`Datos20260320.dat`):**  
Los tres puntos tomados entre 12:10 y 12:14 muestran simetría casi perfecta
($L = 11.3 \to 11.2 \to 11.3$ cm), confirmando que la sombra mínima fue capturada
con fidelidad. El $\chi^2_\text{red} = 0.31$ (con 1 grado de libertad) indica que la
dispersión de los datos es consistente con $\sigma_L = 0.1$ cm. El punto de las 11:57
(~12 minutos antes del mediodía solar) cae fuera de la parábola de los puntos cercanos al
mínimo, comportamiento esperado: la aproximación cuadrática es válida únicamente en una
vecindad del mínimo.

**Conjunto B (`DatosLongitudSombra.dat`):**  
El ajuste parabólico con los 7 puntos produce $a < 0$ (parábola que abre hacia abajo), lo
que es **físicamente inválido** — la longitud de la sombra debe tener un mínimo, no un máximo.
Esto revela que los puntos de las 11:58:40 ($L = 16.0$ cm) y 12:07:27 ($L = 16.9$ cm) son
**valores atípicos** (*outliers*) que no siguen la tendencia esperada. Eliminándolos y
ajustando únicamente los tres puntos de la región 12:09–12:14 se obtiene $a > 0$: el ajuste
es exacto (3 parámetros para 3 datos) con el mediodía solar en 12:12:21 y
$L_\text{min} = 13.69$ cm. Este resultado es coherente con la observación directa
(mínimo medido: 13.7 cm a las 12:12:12).

---

### 8.2 Comparación con el valor aceptado

| Cantidad | Conj. A | Conj. B | Valor aceptado |
|:---|:---:|:---:|:---:|
| $\theta_z$ (°) | $6.291 \pm 0.033$ | $7.646 \pm 0.055$ | $\varphi = 6.267°$ |
| Error relativo $\theta_z$ | $+0.39\,\%$ | $+22.0\,\%$ | — |
| $R_\oplus$ (km) | $6311 \pm 34$ | $5193 \pm 38$ | $6371$ |
| Error relativo $R_\oplus$ | $-0.94\,\%$ | $-18.5\,\%$ | — |
| $C$ (km) | $39\,652 \pm 211$ | $32\,627 \pm 236$ | $40\,030$ |

El **Conjunto A** reproduce el radio de la Tierra con un error de solo $-0.94\,\%$
(el radio medido es $\approx 60$ km menor que el aceptado), resultado extraordinario para
un experimento con instrumentos sencillos.

El **Conjunto B** arroja $\theta_z \approx 7.65°$, un $22\,\%$ mayor que la latitud real
($6.27°$). Un ángulo cenital sobreestimado implica que la sombra fue sistemáticamente más
larga de lo esperado, lo que reduce el radio calculado ($R \approx 5\,193$ km vs. los
$6\,371$ km reales).

---

### 8.3 Fuentes de error y posibles discrepancias

#### a) Inclinación del gnomon
Esta es probablemente la **mayor fuente de error sistemático**. Si el gnomon está
inclinado un ángulo $\Delta\alpha$ hacia el Norte (en dirección al Sol), la sombra se
alarga. Para $H = 102$ cm, una inclinación de $0.5°$ introduce un error de $\approx 0.9$
cm en $L$, equivalente a un error de $\approx 0.5°$ en $\theta_z$ y de $\approx 8\,\%$ en
$R_\oplus$. El exceso de ~1.4° en $\theta_z$ del Conjunto B es compatible con una inclinación
de $\approx 1.4°$ del gnomon, que es difícil de detectar a simple vista sin plomada.

#### b) Horizontalidad de la superficie
Si la superficie de proyección no es horizontal, la longitud medida no corresponde a la
proyección horizontal real. Una pendiente de $1°$ puede introducir $1$–$2$ cm de error en
sombras de $\sim 11$–$14$ cm.

#### c) Penumbra y difusión atmosférica
El Sol tiene un diámetro angular aparente de $\approx 0.53°$, generando una zona de penumbra
al final de la sombra de longitud:
$$\Delta L_\text{penumbra} \approx H\,\tan\!\left(\frac{0.53°}{2}\right) \approx 0.47\,\text{cm}$$
Identificar el borde de la sombra introduce una incertidumbre sistemática que puede superar
los $0.1$ cm declarados, especialmente en días de atmósfera turbia.

#### d) Obstrucción momentánea del Sol (nubes)
El valor atípico de $L = 16.9$ cm a las 12:07:27 en el Conjunto B puede deberse a que
una nube redujo la iluminación directa durante esa medición, haciendo que la sombra
se identificara incorrectamente o que el borde se estimara en un punto diferente.

#### e) Refracción atmosférica
A la elevación solar del mediodía ($\approx 83.7°$), la refracción eleva aparentemente
al Sol $< 0.003°$. Este efecto es completamente despreciable.

#### f) Declinación solar residual
En el equinoccio de 2026 (21:01 UTC del 20 de marzo), al mediodía solar local
($\approx$ 12:09 hora local = 17:09 UTC), la declinación es $|\delta| < 0.02°$, lo que
introduce un error de $< 0.02°$ en $\theta_z$ — despreciable.

#### g) Resolución temporal
Cerca del mínimo, la parábola del Conjunto A es muy plana
($a \approx 0.008$ cm/min$^2$): un error de 1 min introduce solo $\approx 0.008$ cm en
$L_\text{min}$. La resolución temporal no es una fuente importante de error.

#### h) Incertidumbre en la distancia meridional $d$
El valor $M_1 = 110.574$ km/° se toma del elipsoide WGS-84 (estándar geodésico
internacional). La diferencia entre el valor lineal y la integración numérica del
elipsoide es de solo $0.03$ km ($0.004\,\%$), completamente negligible.

---

### 8.4 Conclusiones

1. **El Conjunto A** reproduce el radio de la Tierra con $-0.94\,\%$ de error relativo
   ($R_\oplus = 6\,311 \pm 34$ km), demostrando que el método de Eratóstenes, con cuidado
   experimental, puede alcanzar gran precisión con instrumentos sencillos.

2. **La elección del equinoccio** convirtió al ecuador en el punto de referencia natural
   (ángulo cenital $= 0°$) y eliminó la necesidad de coordinar mediciones simultáneas con
   otra ciudad.

3. **La principal fuente de error sistemático** es la inclinación del gnomon: un desplazamiento
   de $\sim 1°$–$2°$ de la vertical explica el exceso observado de $\approx 1.4°$ en $\theta_z$
   del Conjunto B. El $96.7\,\%$ de la varianza en $\theta_z$ proviene del error en $L$
   (no en $H$), lo que confirma que la medición de la sombra es el cuello de botella.

4. **Los valores atípicos del Conjunto B** (11:58:40 y 12:07:27) distorsionan el ajuste
   parabólico global hasta hacerlo físicamente inválido ($a < 0$). Al restringir el ajuste a
   los tres puntos cercanos al mínimo se recupera un resultado válido, aunque con mayor
   incertidumbre.

5. **Mejoras sugeridas:**
   - Verificar la verticalidad del gnomon con plomada y nivel de burbuja antes de cada sesión.
   - Tomar mediciones cada 1–2 minutos en la ventana de $\pm 15$ min alrededor del mediodía
     predicho.
   - Usar un gnomon con punta afilada o una pantalla con agujero para reducir la penumbra.
   - Aumentar la longitud del gnomon ($H$ mayor → $L$ mayor → menor error relativo en $\theta_z$).
   - Registrar la dirección de la sombra para detectar y corregir la inclinación del gnomon.